# Módulo 04 · Aula 3 — `JOIN`: juntando tabelas

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

A aula 2 mencionou que a normalização reparte os dados em várias tabelas. Esta aula mostra
como juntá-los de volta.

`JOIN` é a operação mais característica do SQL — a que justifica o "relacional" no nome —
e é também onde mais gente erra sem perceber. O erro típico não é sintaxe: a consulta roda,
devolve uma tabela de aparência normal, e o número está errado porque o `JOIN` duplicou
linhas ou perdeu linhas em silêncio.

Por isso esta aula gasta tanto tempo em **conferir** o resultado quanto em produzi-lo. É o
mesmo espírito do `validate=` que você viu no `merge` do módulo 02, aula 4.

Ao final desta aula você vai:

- juntar tabelas com `INNER JOIN` e `LEFT JOIN`, sabendo qual usar;
- entender o que um `NULL` significa no resultado de um `LEFT JOIN`;
- reconhecer quando um `JOIN` **multiplica** linhas, e por quê;
- conferir toda junção antes de confiar nela;
- juntar tabelas de granularidades diferentes — diária com mensal.

**Tempo estimado:** 80 minutos.

### Antes de começar — se você está no Google Colab

Este notebook lê o banco de dados da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "04_SQL"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 140)

conexao = sqlite3.connect("../data/capacitacao.db")


def consultar(sql):
    return pd.read_sql_query(sql, conexao)


print("Conectado.")

## 1. O problema

A tabela `cotacoes` sabe que a PETR4 fechou a R$ 32,50. Ela **não** sabe que PETR4 é a
Petrobras, nem que é do setor de petróleo, nem que é estatal. Isso está em `empresas`.

Qualquer pergunta que misture as duas coisas — *"qual o volume negociado por setor?"* —
exige juntar as tabelas. O elo é o `ticker`, que existe nas duas.

In [ ]:
consultar("SELECT * FROM empresas LIMIT 3")

In [ ]:
consultar("SELECT * FROM cotacoes LIMIT 3")

## 2. `INNER JOIN`

A forma é sempre a mesma:

```sql
FROM       tabela_a
INNER JOIN tabela_b ON tabela_a.chave = tabela_b.chave
```

O `ON` diz **como** as linhas se correspondem. Sem ele, o banco não tem como saber qual
linha de uma casa com qual linha da outra.

In [ ]:
consultar("""
    SELECT
        c.data,
        c.ticker,
        e.empresa,
        e.setor,
        c.fechamento_ajustado
    FROM cotacoes c
    INNER JOIN empresas e ON c.ticker = e.ticker
    WHERE c.data = '2025-12-30'
    ORDER BY e.setor, c.ticker
""")

### Apelidos de tabela

Repare no `cotacoes c` e `empresas e`: são apelidos, e passam a ser obrigatórios na
prática assim que há mais de uma tabela. Sem eles, `ticker` seria ambíguo — existe nas
duas — e a consulta ficaria ilegível.

A convenção é usar a inicial da tabela, ou uma abreviação curta e óbvia.

> **`INNER JOIN` ou só `JOIN`?** São a mesma coisa: `INNER` é o padrão quando você
> escreve apenas `JOIN`. Escrever `INNER` por extenso é hábito melhor — deixa explícito
> para quem lê que a escolha foi deliberada, e não distração.

## 3. O que o `INNER` significa: linhas somem

`INNER JOIN` devolve **só as linhas que casaram dos dois lados**. Se um ticker existe em
`cotacoes` e não em `empresas`, ele desaparece do resultado — sem aviso.

No nosso banco isso não acontece, porque a chave estrangeira impede. Mas em bases reais
acontece o tempo todo, e a única defesa é conferir. Vamos criar o problema de propósito
para ver o efeito.

In [ ]:
# Uma tabela temporária com apenas 3 das 8 empresas
consultar("""
    WITH empresas_parciais AS (
        SELECT * FROM empresas WHERE setor = 'Financeiro'
    )
    SELECT
        COUNT(*) AS linhas_no_resultado
    FROM cotacoes c
    INNER JOIN empresas_parciais e ON c.ticker = e.ticker
""")

3.738 linhas em vez de 9.968. Cinco papéis inteiros sumiram, e nada no resultado
denuncia isso: você recebe uma tabela bem-formada, com colunas certas, e conclui coisas
sobre um mercado que perdeu 62% dos dados.

**É por isso que todo `JOIN` precisa de conferência.** Voltaremos a isso na seção 6.

> O `WITH ... AS (...)` que apareceu aí é uma **CTE**, assunto da aula 4. Por ora, leia
> como "uma tabela temporária que só existe durante esta consulta".

## 4. `LEFT JOIN`: preservar a tabela da esquerda

`LEFT JOIN` mantém **todas** as linhas da tabela à esquerda. Quando não há correspondência
à direita, as colunas da direita vêm como `NULL`.

É a escolha certa quando a tabela da esquerda é a que você quer descrever, e a da direita
é um complemento que pode faltar.

In [ ]:
# Todos os clientes, com uma tabela de metas que só cobre alguns perfis
consultar("""
    WITH metas AS (
        SELECT 'Conservador' AS perfil, 500  AS meta_aporte
        UNION ALL
        SELECT 'Moderado',              1500
    )
    SELECT
        c.perfil_investidor,
        COUNT(*)         AS clientes,
        m.meta_aporte
    FROM clientes c
    LEFT JOIN metas m ON c.perfil_investidor = m.perfil
    GROUP BY c.perfil_investidor, m.meta_aporte
    ORDER BY clientes DESC
""")

Duas coisas para ler nesse resultado:

**Os `None` na coluna `meta_aporte`** são os `NULL` do `LEFT JOIN`: perfis que não têm meta
cadastrada. A informação "não existe meta para este perfil" é preservada — com `INNER JOIN`
essas linhas simplesmente sumiriam.

**As categorias inconsistentes reaparecem.** `CONSERVADOR`, `conservador` e
`'  Conservador '` não casaram com `'Conservador'`, porque o `JOIN` compara texto exato. É
o mesmo defeito da base que você limpou no módulo 02 — e aqui ele se manifesta como meta
faltando, não como categoria duplicada. **Junção sobre dados sujos produz `NULL` que parece
dado faltante mas é dado sujo.**

### Usando o `LEFT JOIN` para *encontrar* o que não casa

Existe um truque que vale ouro: `LEFT JOIN` + `WHERE ... IS NULL` devolve exatamente as
linhas que **não** têm correspondência. É a forma padrão de auditar uma junção.

In [ ]:
consultar("""
    SELECT DISTINCT c.perfil_investidor
    FROM clientes c
    LEFT JOIN (
        SELECT 'Conservador' AS perfil UNION ALL SELECT 'Moderado'
    ) m ON c.perfil_investidor = m.perfil
    WHERE m.perfil IS NULL
    ORDER BY c.perfil_investidor
""")

Ali está a lista dos valores problemáticos, de graça. Antes de rodar qualquer
`JOIN` importante, faça essa consulta: ela responde "o que eu vou perder?".

## 5. Os outros tipos

| Tipo | Mantém | Quando usar |
|---|---|---|
| `INNER JOIN` | só o que casa dos dois lados | o padrão, quando a correspondência é garantida |
| `LEFT JOIN` | tudo da esquerda | a esquerda é o objeto da análise |
| `RIGHT JOIN` | tudo da direita | raro — basta inverter a ordem e usar `LEFT` |
| `FULL OUTER JOIN` | tudo dos dois lados | comparar duas fontes que deveriam bater |
| `CROSS JOIN` | todas as combinações | gerar grades (todo mês × todo papel) |

Na prática, `INNER` e `LEFT` cobrem quase tudo o que você vai escrever.

> **Versão do SQLite.** `RIGHT JOIN` e `FULL OUTER JOIN` só existem no SQLite a partir da
> versão 3.39 (2022). Em ambiente antigo eles falham — mais um motivo para preferir `LEFT`,
> que funciona em qualquer lugar e faz o mesmo trabalho com as tabelas invertidas.

In [ ]:
print("Versão do SQLite:", sqlite3.sqlite_version)

## 6. A conferência que ninguém faz

Esta é a seção mais importante da aula.

Um `JOIN` pode errar de duas maneiras opostas, e **as duas produzem uma tabela de aparência
perfeitamente normal**:

| Erro | Sintoma | Causa |
|---|---|---|
| **perdeu linhas** | resultado menor que o esperado | `INNER JOIN` com chave que não casa |
| **multiplicou linhas** | resultado maior que o esperado | a chave se repete do lado direito |

A defesa é uma só: **conte antes e conte depois.**

In [ ]:
consultar("""
    SELECT
        (SELECT COUNT(*) FROM cotacoes)                  AS antes,
        (SELECT COUNT(*)
           FROM cotacoes c
           INNER JOIN empresas e ON c.ticker = e.ticker) AS depois
""")

9.968 antes, 9.968 depois. Nada foi perdido e nada foi duplicado — este `JOIN` é
confiável.

Esse é o equivalente ao `validate="m:1"` que o pandas oferece no `merge`. O pandas levanta
exceção se a garantia falhar; o SQL não tem esse recurso, então a conferência é sua.

### O `JOIN` que multiplica

Se a chave se repete do lado direito, cada linha da esquerda casa com **várias** da
direita, e o resultado incha. Quando isso acontece com uma coluna que você vai somar, o
total sai inflado — e continua parecendo um número plausível.

Vamos construir o caso: uma tabela de metas com o mesmo perfil repetido duas vezes.

In [ ]:
consultar("""
    WITH metas_duplicadas AS (
        SELECT 'Conservador' AS perfil, 500  AS meta
        UNION ALL
        SELECT 'Conservador',            900   -- a MESMA chave, de novo
    )
    SELECT
        (SELECT COUNT(*) FROM clientes WHERE perfil_investidor = 'Conservador') AS antes,
        COUNT(*)                                                                AS depois
    FROM clientes c
    INNER JOIN metas_duplicadas m ON c.perfil_investidor = m.perfil
""")

O dobro. Cada cliente conservador virou duas linhas.

Se a consulta seguinte fosse `SUM(aporte_mensal)`, o total de aportes viria **dobrado** —
e nada, absolutamente nada no resultado indicaria o problema. Este é o erro de SQL que mais
custa caro em ambiente profissional, e a única proteção é o hábito de contar antes e
depois.

> **A pergunta a fazer antes de todo `JOIN`:** *a chave é única do lado direito?* Se você
> não sabe, descubra:
> ```sql
> SELECT chave, COUNT(*) FROM tabela GROUP BY chave HAVING COUNT(*) > 1
> ```
> Se isso devolver alguma linha, seu `JOIN` vai multiplicar.

In [ ]:
# Aplicando a receita ao nosso banco: a chave de `empresas` é única?
consultar("""
    SELECT ticker, COUNT(*) AS vezes
    FROM empresas
    GROUP BY ticker
    HAVING COUNT(*) > 1
""")

Vazio — a chave é única, o `JOIN` é seguro. Um resultado vazio aqui é a melhor
notícia possível.

## 7. `JOIN` com agregação

Aqui o `JOIN` começa a pagar: perguntas que nenhuma das tabelas responde sozinha.

In [ ]:
consultar("""
    SELECT
        e.setor,
        COUNT(DISTINCT e.ticker)            AS papeis,
        ROUND(AVG(c.fechamento_ajustado), 2) AS preco_medio,
        SUM(c.volume)                       AS volume_total
    FROM cotacoes c
    INNER JOIN empresas e ON c.ticker = e.ticker
    WHERE c.data >= '2025-01-01'
    GROUP BY e.setor
    ORDER BY volume_total DESC
""")

Repare no `COUNT(DISTINCT e.ticker)`. Um `COUNT(*)` aí contaria **pregões**, não
papéis — o setor Financeiro tem 3 papéis e uns 750 pregões. Depois de um `JOIN`, sempre
pergunte a si mesmo: *uma linha deste resultado é o quê?* Aqui, cada linha é um par
(papel, pregão), e é por isso que contar papéis exige `DISTINCT`.

## 8. Granularidades diferentes: diário × mensal

Este é o `JOIN` que separa quem entendeu de quem decorou.

`cotacoes` é **diária**. `indicadores` é **mensal**. Não existe chave em comum: não há
uma linha de `indicadores` para o dia 15 de março.

A solução tem duas etapas, e a ordem é o que importa:

1. **agregar a tabela diária para mensal** — reduzir a granularidade fina até a grossa;
2. **só então juntar**, agora que as duas falam da mesma unidade de tempo.

Tentar juntar antes de agregar é o erro clássico: como cada mês de indicadores casaria com
uns 21 pregões, os valores mensais seriam contados 21 vezes.

In [ ]:
consultar("""
    WITH mensal AS (
        SELECT
            ticker,
            strftime('%Y-%m', data)             AS ano_mes,
            ROUND(AVG(fechamento_ajustado), 2)  AS preco_medio,
            COUNT(*)                            AS pregoes
        FROM cotacoes
        WHERE ticker = 'PETR4'
        GROUP BY ticker, ano_mes
    )
    SELECT
        m.ano_mes,
        m.pregoes,
        m.preco_medio,
        i.ipca_mes_pct,
        i.selic_mes_pct,
        i.dolar_medio
    FROM mensal m
    INNER JOIN indicadores i
        ON m.ano_mes = strftime('%Y-%m', i.data)
    ORDER BY m.ano_mes DESC
    LIMIT 12
""")

Agora sim: uma linha por mês, com o preço médio do papel ao lado dos
indicadores daquele mês. É exatamente a tabela que o ciclo 3 do notebook de EDA construiu
em pandas, no módulo 03 — e é a base para perguntar se preço e Selic andam juntos.

**A conferência, sempre:** 60 meses de indicadores, e a consulta deveria devolver 60 linhas
para um papel.

In [ ]:
consultar("""
    WITH mensal AS (
        SELECT ticker, strftime('%Y-%m', data) AS ano_mes
        FROM cotacoes WHERE ticker = 'PETR4'
        GROUP BY ticker, ano_mes
    )
    SELECT
        (SELECT COUNT(*) FROM indicadores)  AS meses_de_indicadores,
        (SELECT COUNT(*) FROM mensal)       AS meses_de_cotacoes,
        (SELECT COUNT(*) FROM mensal m
           INNER JOIN indicadores i ON m.ano_mes = strftime('%Y-%m', i.data)) AS depois_do_join
""")

60, 60, 60. A junção é um-para-um e não perdeu nem duplicou nada.

## 9. Juntando uma tabela com ela mesma

Um `self-join` é um `JOIN` de uma tabela consigo própria, com apelidos diferentes. Serve
para comparar linhas da mesma tabela — por exemplo, dois papéis no mesmo dia.

In [ ]:
consultar("""
    SELECT
        a.data,
        ROUND(a.fechamento_ajustado, 2) AS petr4,
        ROUND(b.fechamento_ajustado, 2) AS vale3,
        ROUND(a.fechamento_ajustado - b.fechamento_ajustado, 2) AS diferenca
    FROM cotacoes a
    INNER JOIN cotacoes b
        ON a.data = b.data          -- mesmo pregão
    WHERE a.ticker = 'PETR4'
      AND b.ticker = 'VALE3'
    ORDER BY a.data DESC
    LIMIT 5
""")

> Comparar uma linha com a **anterior** — o retorno de um dia para o outro — também
> dá para fazer com self-join, mas é desajeitado. A aula 4 mostra a ferramenta feita para
> isso: a função de janela `LAG`.

## 10. Conferindo contra o pandas

> *Qual o retorno médio diário por setor em 2025?*

In [ ]:
via_sql = consultar("""
    WITH com_retorno AS (
        SELECT
            c.ticker,
            e.setor,
            c.data,
            c.fechamento_ajustado
        FROM cotacoes c
        INNER JOIN empresas e ON c.ticker = e.ticker
        WHERE c.data >= '2025-01-01'
    )
    SELECT
        setor,
        COUNT(*)                             AS observacoes,
        ROUND(AVG(fechamento_ajustado), 2)   AS preco_medio
    FROM com_retorno
    GROUP BY setor
    ORDER BY setor
""")
via_sql

In [ ]:
acoes = pd.read_csv("../data/acoes_b3.csv", parse_dates=["data"])
empresas = pd.read_csv("../data/empresas_b3.csv")

juntas = acoes.merge(empresas, on="ticker", how="inner", validate="m:1")
via_pandas = (
    juntas[juntas["data"] >= "2025-01-01"]
    .groupby("setor")["fechamento_ajustado"]
    .agg(observacoes="count", preco_medio="mean")
    .round({"preco_medio": 2})
    .reset_index()
    .sort_values("setor")
    .reset_index(drop=True)
)
via_pandas

In [ ]:
import numpy as np

print("Mesmo resultado:",
      np.array_equal(via_sql["observacoes"].values, via_pandas["observacoes"].values)
      and np.allclose(via_sql["preco_medio"].values, via_pandas["preco_medio"].values))

Repare no `validate="m:1"` da versão pandas: é a garantia de que a junção é
muitos-para-um, e o pandas levanta exceção se não for. **O SQL não tem equivalente** — a
conferência manual da seção 6 é o substituto, e é por isso que ela precisa virar hábito.

## 11. O mapa, atualizado

| Pergunta | pandas | SQL |
|---|---|---|
| juntar tabelas | `.merge(..., how="inner")` | `INNER JOIN ... ON` |
| preservar a esquerda | `.merge(..., how="left")` | `LEFT JOIN ... ON` |
| garantir a cardinalidade | `validate="m:1"` | contar antes e depois, na mão |
| achar o que não casou | `indicator=True` | `LEFT JOIN` + `WHERE ... IS NULL` |
| contar categorias distintas | `.nunique()` | `COUNT(DISTINCT ...)` |

## 12. Recapitulando

- A normalização reparte os dados; o `JOIN` junta de volta, e o `ON` diz como.
- **`INNER JOIN` descarta o que não casa; `LEFT JOIN` preserva a esquerda e preenche com
  `NULL`.** Escolher errado perde linhas em silêncio.
- Apelidos de tabela deixam de ser conforto e viram necessidade a partir da segunda tabela.
- `NULL` depois de um `LEFT JOIN` pode significar "não existe" **ou** "a chave estava
  suja". São coisas diferentes com o mesmo sintoma.
- `LEFT JOIN` + `WHERE ... IS NULL` lista exatamente o que não casou. Use antes, não
  depois.
- **Todo `JOIN` precisa de conferência: conte antes, conte depois.** Perder linhas e
  multiplicar linhas produzem tabelas igualmente normais à primeira vista.
- Antes de juntar, pergunte se a chave é única do lado direito. Se não for, o resultado
  vai multiplicar.
- Para juntar granularidades diferentes, **agregue primeiro, junte depois**.
- Depois de um `JOIN`, sempre reformule: *uma linha deste resultado é o quê?*

**Próxima aula:** subconsultas, CTEs e funções de janela — as ferramentas que transformam
consultas grandes em consultas legíveis, e que permitem calcular retorno e média móvel
dentro do próprio banco.

In [ ]:
conexao.close()
print("Conexão fechada.")